In [1]:
# ============================================================
# Install Required Packages
# ============================================================
!pip -q install transformers
!pip -q install datasets
!pip -q install huggingface_hub
!pip -q install jiwer
!pip -q install tqdm
!pip -q install pandas
!pip -q install torchaudio
!pip -q install onnxruntime
!pip -q install onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 80.1 MB/s eta 0:00:00


In [2]:
# ============================================================
# Import Required Libraries
# ============================================================
# Standard Library
import time

# Data Handling
import pandas as pd

# Deep Learning
import torch
import torchaudio

# Hugging Face
from transformers import AutoModel
from datasets import load_dataset
from huggingface_hub import login

# Evaluation
from jiwer import wer

# Progress Bar
from tqdm.auto import tqdm

In [3]:
# ============================================================
# Hugging Face Login
# ============================================================
from huggingface_hub import login
login()

In [5]:
# ============================================================
# Load IndicConformer Model
# ============================================================
MODEL_NAME = "ai4bharat/indic-conformer-600m-multilingual"

print("=" * 60)
print("Loading IndicConformer Model...")
print("=" * 60)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("\n✅ Model loaded successfully!")

Loading IndicConformer Model...
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate

✅ Model loaded successfully!


In [6]:
# ============================================================
# Configuration
# ============================================================
# Model Configuration
MODEL_NAME = "ai4bharat/indic-conformer-600m-multilingual"
DECODER = "rnnt"
# Dataset Configuration
DATASET_NAME = "ai4bharat/kathbath"
DATASET_SPLIT = "valid"

# Audio Configuration
TARGET_SAMPLE_RATE = 16000

# Benchmark Configuration
NUM_SAMPLES = 100

# Languages to Benchmark
LANGUAGES = {
    "Hindi": "hindi",
    "Telugu": "telugu",
    "Tamil": "tamil",
    "Malayalam": "malayalam",
    "Kannada": "kannada",
    "Punjabi": "punjabi",
    "Gujarati": "gujarati",
    "Bengali": "bengali",
    "Marathi": "marathi"
}

In [7]:
# ============================================================
# Audio Preprocessing
# ============================================================
def preprocess_audio(sample):
    """
    Converts a dataset audio sample into a mono 16 kHz PyTorch tensor.
    """
    audio = sample["audio_filepath"]
    waveform = torch.tensor(audio["array"]).unsqueeze(0)
    sample_rate = audio["sampling_rate"]
    # Convert to mono if required
    waveform = torch.mean(waveform, dim=0, keepdim=True)
    # Resample to target sample rate
    if sample_rate != TARGET_SAMPLE_RATE:
        resampler = torchaudio.transforms.Resample(sample_rate,TARGET_SAMPLE_RATE)
        waveform = resampler(waveform)
    return waveform

In [8]:
# ============================================================
# ASR Inference
# ============================================================
def transcribe_sample(sample):
    """
    Runs IndicConformer inference on one audio sample.
    Returns transcription and inference time.
    """
    waveform = preprocess_audio(sample)
    start_time = time.time()
    prediction = model(
        waveform,
        sample["lang"],
        DECODER
    )
    inference_time = time.time() - start_time
    return prediction, inference_time

In [9]:
# ============================================================
# WER Calculation
# ============================================================
def calculate_wer(reference, prediction):
    return wer(reference, prediction)

In [10]:
# ============================================================
# Benchmark One Language
# ============================================================
def benchmark_language(language_name, dataset_config):
    print(f"\nBenchmarking {language_name}")
    dataset = load_dataset(
        DATASET_NAME,
        dataset_config,
        split=DATASET_SPLIT,
        streaming=True
    )
    summary = []
    details = []
    total_wer = 0
    total_time = 0
    processed = 0
    for sample in tqdm(dataset, total=NUM_SAMPLES):
        if processed == NUM_SAMPLES:
            break
        try:
            prediction, inference_time = transcribe_sample(sample)
            sample_wer = calculate_wer(sample["text"],prediction)
            details.append({
                "language": language_name,
                "file_name": sample["fname"],
                "duration": sample["duration"],
                "ground_truth": sample["text"],
                "prediction": prediction,
                "wer": sample_wer,
                "inference_time_sec": round(inference_time, 3)})
            total_wer += sample_wer
            total_time += inference_time
            processed += 1
        except Exception as e:
            print(f"Skipped {sample['fname']} : {e}")
    summary.append({
        "language": language_name,
        "model_name": MODEL_NAME,
        "decoder": DECODER,
        "dataset": DATASET_NAME,
        "split": DATASET_SPLIT,
        "num_samples": processed,
        "avg_wer": round(total_wer / processed, 4),
        "avg_inference_time_sec": round(total_time / processed,4)
    })
    return summary, details

In [11]:
# ============================================================
# Run Benchmark for All Languages
# ============================================================
all_summary = []
all_details = []
print("=" * 60)
print("Starting ASR Benchmark...")
print("=" * 60)

for language_name, dataset_config in LANGUAGES.items():
    summary, details = benchmark_language(language_name,dataset_config)
    all_summary.extend(summary)
    all_details.extend(details)

print("\n" + "=" * 60)
print("Benchmark Completed Successfully!")
print("=" * 60)

Starting ASR Benchmark...

Benchmarking Hindi


README.md:   0%|          | 0.00/10.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

...


Benchmarking Marathi


Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]


Benchmark Completed Successfully!


In [12]:
# ============================================================
# Create Results Directory and Save CSV Files
# ============================================================
import os
RESULTS_DIR = 'results'
# Create results directory if it doesn't exist
os.makedirs(RESULTS_DIR, exist_ok=True)

# Create DataFrames
summary_df = pd.DataFrame(all_summary)
details_df = pd.DataFrame(all_details)

# Save CSV files
summary_path = os.path.join(RESULTS_DIR, "benchmark_summary.csv")
details_path = os.path.join(RESULTS_DIR, "benchmark_details.csv")

summary_df.to_csv(summary_path, index=False, encoding="utf-8")
details_df.to_csv(details_path, index=False, encoding="utf-8")

print("=" * 60)
print("Results saved successfully!")
print("=" * 60)
print(f"Summary : {summary_path}")
print(f"Details : {details_path}")

Results saved successfully!
Summary : results/benchmark_summary.csv
Details : results/benchmark_details.csv


In [13]:
# ============================================================
# Download Benchmark Results
# ============================================================
from google.colab import files
files.download(summary_path)
files.download(details_path)
print(" Download completed.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Download completed.


In [14]:
summary_df

,language,model_name,decoder,dataset,split,num_samples,avg_wer,avg_inference_time_sec
0,Hindi,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.0765,1.4359
1,Telugu,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.2209,2.1196
2,Tamil,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.1924,1.8521
3,Malayalam,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.3150,2.9976
4,Kannada,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.1312,2.1941
5,Punjabi,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.0936,1.4495
6,Gujarati,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.0988,1.8818
7,Bengali,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.0968,2.0139
8,Marathi,ai4bharat/indic-conformer-600m-multilingual,rnnt,ai4bharat/kathbath,valid,100,0.0958,1.8452


In [15]:
details_df

,language,file_name,duration,ground_truth,prediction,wer,inference_time_sec
0,Hindi,844424930524026-252-f.m4a,4.017062,हमने ओबीसी तबके के बच्चों के लिए उच्च शिक्षा म...,हमने ओबीसी तबके के बच्चों के लिए उच्च शिक्षा म...,0.000000,4.277
1,Hindi,844424930612535-939-f.m4a,4.806563,उसको छोड़कर लोग इस प्रकार के मुद्दो पर चर्चा क...,उसको छोड़कर लोग इस प्रकार के मुद्दों पर चर्चा ...,0.166667,2.364
2,Hindi,844424930869370-252-f.m4a,3.738437,उन्होंने कहा कि अतिथि अध्यापको को हटाना न्यायस...,उन्होंने कहा कि अतिथि अध्यापकों को हटाना न्याय...,0.100000,2.065
3,Hindi,844424931201045-939-f.m4a,4.853000,चीन में मोबाइल फोन की लत एक युवती को बड़ी मुश्...,चीन में मोबाइल फोन की लत एक युवती को बड़ी मुश्...,0.000000,2.286
4,Hindi,844424931205944-164-f.m4a,3.784875,मगर साथ में मनपसंद साथी हो तो मेहनत भी रास आती है,मगर साथ में मनपसंद साथी हो तो मेहनत भी रास आती है,0.000000,1.098
...,...,...,...,...,...,...,...
895,Marathi,844424932789495-427-f.m4a,7.407188,जेथे मृत व्यक्तीला जिवंत आणि जिवंत व्यक्ती मृत...,जेथे मृत व्यक्तीला जिवंत आणि जिवंत व्यक्ती मृत...,0.000000,1.983
896,Marathi,844424931031673-1106-f.m4a,5.874688,याकामी दिव्यांग आधार फाउंडेशनचे गणेश शेटे व मह...,याकामी दिव्यांग आधार फाऊंडेशनचे गणेश शेटे व मह...,0.076923,1.648
897,Marathi,844424930882706-427-f.m4a,5.224500,माझ्या आईला जिथं जाळलं तिथं पाण्यानं लहानसा खड...,माझ्या आईला जिथं जाळलं तिथं पाण्यानं लहानसा खड...,0.000000,1.912
898,Marathi,844424932851338-1106-f.m4a,6.455188,त्यावेळी न्यायालयाने कोणताही अहवाल सार्वजनिक क...,त्यावेळी न्यायालयाने कोणताही अहवाल सार्वजनिक क...,0.000000,2.223
